# AirShift — Data Cleaning

This notebook prepares the raw air quality data for analysis and machine learning.

The cleaning process focuses on:

* Combining data from all monitoring stations
* Creating a consistent datetime column
* Validating data types and values
* Handling missing values carefully
* Checking duplicates and temporal consistency
* Saving the cleaned dataset for the next stage of the AirShift pipeline

No raw data will be overwritten during the cleaning process.

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np

In [3]:
# Project directories
DATA_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../data/processed")

# Create processed-data directory if it does not exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data directory:")
print(DATA_DIR.resolve())

print("\nProcessed data directory:")
print(OUTPUT_DIR.resolve())

Raw data directory:
C:\Users\HP\Desktop\Projects\airshift\data\raw

Processed data directory:
C:\Users\HP\Desktop\Projects\airshift\data\processed


In [4]:
files = sorted(DATA_DIR.glob("*.csv"))

print(f"Number of CSV files: {len(files)}")

for file in files:
    print(file.name)

Number of CSV files: 12
PRSA_Data_Aotizhongxin_20130301-20170228.csv
PRSA_Data_Changping_20130301-20170228.csv
PRSA_Data_Dingling_20130301-20170228.csv
PRSA_Data_Dongsi_20130301-20170228.csv
PRSA_Data_Guanyuan_20130301-20170228.csv
PRSA_Data_Gucheng_20130301-20170228.csv
PRSA_Data_Huairou_20130301-20170228.csv
PRSA_Data_Nongzhanguan_20130301-20170228.csv
PRSA_Data_Shunyi_20130301-20170228.csv
PRSA_Data_Tiantan_20130301-20170228.csv
PRSA_Data_Wanliu_20130301-20170228.csv
PRSA_Data_Wanshouxigong_20130301-20170228.csv


## 1. Load and Combine Station Data

The raw dataset contains hourly air quality measurements from 12 monitoring stations.

We combine the station files into a single dataset while preserving the `station` column. This provides one consistent table for subsequent cleaning and feature engineering.

The original raw files are kept unchanged.

In [5]:
station_data = []

for file in files:
    df = pd.read_csv(file)
    station_data.append(df)

df = pd.concat(station_data, ignore_index=True)

print("Combined dataset shape:", df.shape)
print("\nNumber of stations:", df["station"].nunique())
print("\nStations:")
print(sorted(df["station"].unique()))

Combined dataset shape: (420768, 18)

Number of stations: 12

Stations:
['Aotizhongxin', 'Changping', 'Dingling', 'Dongsi', 'Guanyuan', 'Gucheng', 'Huairou', 'Nongzhanguan', 'Shunyi', 'Tiantan', 'Wanliu', 'Wanshouxigong']


## 2. Create Datetime

The raw dataset stores time information in four separate columns: `year`, `month`, `day`, and `hour`.

We combine these columns into a single `datetime` column to provide a consistent representation of time.

This will make the data easier to work with during time-series analysis, temporal feature engineering, sorting, and the definition of historical and future conditions for the AirShift early-warning task.


In [6]:
df["datetime"] = pd.to_datetime(
    df[["year", "month", "day", "hour"]]
)

print("Datetime column created successfully.")

print("\nFirst timestamps:")
print(df["datetime"].head())

print("\nLast timestamps:")
print(df["datetime"].tail())

Datetime column created successfully.

First timestamps:
0   2013-03-01 00:00:00
1   2013-03-01 01:00:00
2   2013-03-01 02:00:00
3   2013-03-01 03:00:00
4   2013-03-01 04:00:00
Name: datetime, dtype: datetime64[us]

Last timestamps:
420763   2017-02-28 19:00:00
420764   2017-02-28 20:00:00
420765   2017-02-28 21:00:00
420766   2017-02-28 22:00:00
420767   2017-02-28 23:00:00
Name: datetime, dtype: datetime64[us]


## 3. Validate Data Types

Before handling missing values or invalid observations, we verify that each column has an appropriate data type.

Correct data types are important because they determine how the variables can be analyzed and processed during the cleaning and machine learning stages.

In particular, the temporal columns should be numeric, the measurement variables should be numeric, and the `station` and `wd` columns should be treated as categorical variables.


In [7]:
df.dtypes

No                   int64
year                 int64
month                int64
day                  int64
hour                 int64
PM2.5              float64
PM10               float64
SO2                float64
NO2                float64
CO                 float64
O3                 float64
TEMP               float64
PRES               float64
DEWP               float64
RAIN               float64
wd                     str
WSPM               float64
station                str
datetime    datetime64[us]
dtype: object

### Data Types Validation Findings

The data types are appropriate for the current cleaning stage.

* Temporal components (`year`, `month`, `day`, and `hour`) are stored as integers.
* Air quality and meteorological measurements are stored as numeric (`float64`) variables.
* `wd` and `station` are stored as string variables and will be treated as categorical features when needed.
* The newly created `datetime` column is stored as a datetime type suitable for time-series operations.

No data type conversions are required at this stage.